# `agora.equities` — Quickstart Examples

A tour of the API-first equities surface in [agora](../README.md).
`agora` is a thin client over the Massive (Polygon) REST API; every
helper here makes a live API call. Downstream packages own caching
and storage.

**Requires** `MASSIVE_API_KEY` in your environment (or `.env` file).

## Contents

1. [Setup](#1-Setup)
2. [Market data — historical & live](#2-Market-data)
3. [Reference / universe discovery](#3-Reference)
4. [Corporate actions — dividends & splits](#4-Corporate-actions)
5. [Fundamentals — financial statements & ratios](#5-Fundamentals)
6. [Short data — interest, volume, floats](#6-Short-data)
7. [ETF analysis — constituents, flows, profiles](#7-ETF-analysis)
8. [Classification — industry & sector](#8-Classification)


## 1. Setup

The package auto-loads `MASSIVE_API_KEY` from `.env` via
`MassiveConfig.from_env()`. The first call to `get_client()` (or
implicit calls inside helper functions) instantiates a process-wide
singleton.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from agora import equities, get_client

# Sanity-check: the client should resolve from your env config.
client = get_client()
print(f"client: {type(client).__name__}, base_url: {client.config.base_url}")


## 2. Market data

### 2.1 Single-ticker close prices, period-style

`get_daily_prices` accepts a `period` shortcut (`"1y"`, `"6mo"`,
`"ytd"`, etc.) or explicit `start` / `end` dates.


In [ ]:
prices = equities.get_daily_prices("AAPL", period="1y")
prices.tail()


### 2.2 Basket of tickers, multi-field

Pass a sequence of tickers to get a date-indexed matrix. Pass a
sequence of fields for a `(field, ticker)` `MultiIndex`.


In [ ]:
basket = equities.get_daily_prices(
    ["AAPL", "MSFT", "NVDA"],
    period="6mo",
    fields=("close", "volume"),
)
basket.tail()


In [ ]:
# Quick correlation matrix on returns
returns = equities.get_daily_returns(["AAPL", "MSFT", "NVDA", "SPY"], period="1y")
returns.corr().round(3)


### 2.3 Cross-section for one date

`get_daily_grouped(date)` makes a single bulk API call returning every
active ticker for that date. Faster than per-ticker for wide queries.
Pass `tickers=` to filter client-side.


In [ ]:
day = equities.get_daily_grouped("2024-01-03", tickers=["AAPL", "MSFT", "NVDA", "SPY", "QQQ"])
day


### 2.4 Live snapshot (current state)

`get_snapshot` fast-paths a single ticker and uses the bulk endpoint
when given a basket. Returns one row per ticker with intraday +
prev-day fields.


In [ ]:
snap = equities.get_snapshot(["AAPL", "MSFT", "NVDA"])
snap[["ticker", "day_close", "day_volume", "prev_close", "todays_change_pct"]]


### 2.5 Live tick — last trade and last quote

For sub-second latency on a single ticker.


In [ ]:
trade = equities.get_last_trade("AAPL")
print(f"last trade: ${trade['price']} × {trade['size']} @ {trade.get('sip_timestamp_utc')}")

quote = equities.get_last_quote("AAPL")
print(f"NBBO:  ${quote['bid_price']} × {quote['bid_size']}  /  ${quote['ask_price']} × {quote['ask_size']}")


### 2.6 Market state

When is the market open? When are upcoming holidays?


In [ ]:
status = equities.get_market_status()
print(f"market: {status['market']}, after_hours: {status['after_hours']}, server: {status['server_time']}")

equities.get_market_holidays().head(10)


### 2.7 Previous close

Per-ticker previous trading day's bar — convenient for pre-market
context.


In [ ]:
equities.get_previous_close(["AAPL", "MSFT", "NVDA"])


## 3. Reference

### 3.1 Universe discovery — `get_tickers`

Filter by market, type, active status, or arbitrary date for a
point-in-time universe.


In [ ]:
# All active US ETFs
etfs = equities.get_tickers(market="stocks", type="ETF")
print(f"{len(etfs):,} active ETFs")
etfs[["ticker", "name", "primary_exchange"]].head()


### 3.2 Per-ticker rich profile — `get_ticker_details`

One API call returns ~25 fields per ticker — market cap, shares,
SIC code, exchange, list date, description, etc.


In [ ]:
profile = equities.get_ticker_details(["AAPL", "MSFT", "NVDA"])
profile[[
    "ticker", "name", "primary_exchange", "market_cap",
    "share_class_shares_outstanding", "sic_description", "list_date",
]]


### 3.3 Related tickers, exchanges, types

Three small reference utilities:


In [ ]:
# Polygon's similarity graph
related = equities.get_related_tickers("AAPL")
print("Related to AAPL:", related["ticker"].tolist())

# Exchanges catalog (one row per venue)
exch = equities.get_exchanges(asset_class="stocks")
print(f"\n{len(exch)} stock exchanges")
print(exch[["mic", "name", "type"]].head())

# Ticker type codes (CS, ETF, ADRC, etc.)
types_ = equities.get_ticker_types(asset_class="stocks")
print(f"\n{len(types_)} ticker types")
types_


## 4. Corporate actions

### 4.1 Dividends


In [ ]:
divs = equities.cax.get_dividends("AAPL", start="2020-01-01")
print(f"{len(divs)} AAPL dividend events since 2020")
divs.tail(8)


### 4.2 Splits


In [ ]:
splits = equities.cax.get_splits("AAPL")
splits  # AAPL's full split history


## 5. Fundamentals

### 5.1 Income statement (annual)


In [ ]:
income = equities.fundamentals.get_income_statements("AAPL", timeframe="annual")
print(f"{len(income)} annual income statements")
income[[
    "tickers", "period_end", "fiscal_year",
    "revenue", "gross_profit", "operating_income", "ebitda",
    "diluted_earnings_per_share",
]].sort_values("period_end").tail()


### 5.2 Balance sheet & cash flow

Pull all three statements in parallel for a thorough fundamental view.


In [ ]:
bs = equities.fundamentals.get_balance_sheets("AAPL", timeframe="annual")
cf = equities.fundamentals.get_cash_flow_statements("AAPL", timeframe="annual")

# Trend: total assets and net cash from operations
bs_summary = bs[["period_end", "fiscal_year"]].copy()
# (Available columns vary by what Polygon reports per period;
# project the ones you actually need.)
print("Balance sheet columns sample:", [c for c in bs.columns[:15]])
print("\nCash flow columns sample: ", [c for c in cf.columns[:10]])


### 5.3 Daily ratios (point-in-time)

Updated daily — P/E, P/B, ev_to_ebitda, dividend_yield, return_on_equity, etc.


In [ ]:
ratios = equities.fundamentals.get_ratios("AAPL")
print(f"{len(ratios)} ratio snapshots")
ratios[[
    "ticker", "date", "price_to_earnings", "price_to_book",
    "ev_to_ebitda", "dividend_yield", "return_on_equity",
]].sort_values("date").tail()


## 6. Short data

### 6.1 Short interest (bi-monthly settlement-date snapshots)


In [ ]:
si = equities.short_data.get_short_interest("AAPL", start="2024-01-01")
si.sort_values("settlement_date").tail()


### 6.2 Daily short volume


In [ ]:
sv = equities.short_data.get_short_volume("AAPL", start="2024-12-01")
sv[["ticker", "date", "short_volume", "total_volume", "short_volume_ratio"]].head(10)


### 6.3 Free-float


In [ ]:
floats = equities.short_data.get_floats("AAPL")
floats


## 7. ETF analysis

### 7.1 SPY constituents and weights


In [ ]:
holdings = equities.etf.get_constituents("SPY")
print(f"SPY has {len(holdings)} reported constituents")
holdings.sort_values("weight", ascending=False)[
    ["constituent_ticker", "constituent_name", "weight", "shares_held"]
].head(15)


### 7.2 Reverse lookup — every ETF that holds AAPL


In [ ]:
aapl_holders = equities.etf.get_constituents(
    constituent_ticker="AAPL",
)
print(f"{len(aapl_holders)} ETFs hold AAPL on the latest reported date")
aapl_holders.sort_values("weight", ascending=False)[
    ["composite_ticker", "weight", "market_value"]
].head(15)


### 7.3 SPY fund flows


In [ ]:
flows = equities.etf.get_fund_flows("SPY", start="2024-01-01")
print(f"{len(flows)} daily flow records")
flows.sort_values("effective_date").tail(10)[
    ["effective_date", "fund_flow", "nav", "shares_outstanding"]
]


### 7.4 SPY profile + analytics + taxonomy

Three lookups for a complete ETF picture.


In [ ]:
profile = equities.etf.get_profiles("SPY")
print("SPY profile sample:")
sample_cols = [c for c in ("composite_ticker", "issuer", "aum", "creation_unit_size",
                            "distribution_frequency") if c in profile.columns]
print(profile.head(1)[sample_cols].T)

analytics = equities.etf.get_analytics("SPY")
sample_cols = [c for c in ("composite_ticker", "risk_total_score", "reward_score",
                            "quant_total_score", "quant_grade") if c in analytics.columns]
print("\nSPY analytics sample:")
print(analytics.head(1)[sample_cols].T)


## 8. Classification

### 8.1 Industry (SIC description) and sector (SIC division)

Both functions are thin wrappers over `get_ticker_details`.


In [ ]:
basket = ["AAPL", "JPM", "XOM", "JNJ", "WMT"]
industry = equities.get_industry(basket)
sector = equities.get_sector(basket)

pd.DataFrame({"industry": industry, "sector": sector})


---

## What's next

- For long-running batch workflows, layer your own caching on top of
  these helpers — `agora.loaders.parquet.FlatFileLoader` is available
  for read-only access to the bulk parquet store populated by
  `python -m agora.download`.
- For the live security master + change log, see
  `python -m agora.download security-master` (documented in
  `dovs/c.download.md`).
- Benzinga-entitled features (`get_major_news`, `get_earnings`)
  remain `NotImplementedError` until the add-on is enabled on
  the Massive account.
